#### Imports

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O

np.random.seed(42)

#### Parameters

In [2]:
N_STUDENTS = 1200
YEARS = [2024, 2025]

countries = ["USA", "India", "Brazil", "Germany", "Kenya", "Peru", "UK"]
education_levels = ["undergrad", "masters", "phd"]

disability_types = [
    "none", "visual", "hearing", "motor",
    "cognitive", "multiple", "prefer_not_to_say"
]

#### Generate applications dataset

In [3]:
def generate_applications():
    data = []
    
    for year in YEARS:
        for i in range(N_STUDENTS):
            student_id = f"{year}_{i}"
            
            disability = np.random.choice(
                disability_types,
                p=[0.75, 0.05, 0.04, 0.04, 0.06, 0.03, 0.03]
            )
            
            has_disability = 0 if disability == "none" else 1
            country = np.random.choice(countries)
            
            # introduce inconsistency
            if country == "USA" and np.random.rand() < 0.2:
                country = np.random.choice(["United States", "US"])
            
            if country == "Brazil" and np.random.rand() < 0.2:
                country = "Brasil"
            
            if country == "UK" and np.random.rand() < 0.2:
                country = "United Kingdom"
            data.append({
                "student_id": student_id,
                "year": year,
                "country": country,
                "gender": np.random.choice(["male", "female", "non_binary"]),
                "education_level": np.random.choice(education_levels),
                "accepted": np.random.choice([0, 1], p=[0.3, 0.7]),
                "has_disability": has_disability,
                "disability_type": disability
            })
    
    df = pd.DataFrame(data)
    
    # introduce messy duplicates
    df = pd.concat([df, df.sample(50)], ignore_index=True)
    
    
    return df

applications = generate_applications()
applications.to_csv("../data/raw/applications.csv", index=False)

applications.head()


,student_id,year,country,gender,education_level,accepted,has_disability,disability_type
0,2024_0,2024,Kenya,non_binary,phd,1,0,none
1,2024_1,2024,India,non_binary,phd,0,0,none
2,2024_2,2024,Germany,non_binary,masters,0,1,motor
3,2024_3,2024,Peru,female,masters,0,0,none
4,2024_4,2024,Germany,female,masters,0,0,none


#### Generate daily survey dataset

In [4]:
def generate_daily_surveys(applications):
    data = []
    
    for _, row in applications.iterrows():
        if row["accepted"] == 0:
            continue
        
        for day in range(1, 16):  # 15-day course
            
            # simulate dropout
            if np.random.rand() < 0.05 * day:
                break
            
            engagement = np.clip(
                np.random.normal(3.5, 1), 1, 5
            )
            
            # slightly lower engagement for some disabilities (realistic bias to analyze)
            if row["has_disability"]:
                engagement -= np.random.normal(0.2, 0.3)
            
            data.append({
                "student_id": row["student_id"],
                "year": row["year"],
                "day": day,
                "attendance": 1,
                "engagement_score": round(engagement, 2),
                "difficulty": np.random.randint(2, 5)
            })
    
    df = pd.DataFrame(data)
    
    # missing data
    df = df.sample(frac=0.95, random_state=42)
    
    return df

daily = generate_daily_surveys(applications)
daily.to_csv("../data/raw/daily_surveys.csv", index=False)

daily.head()

,student_id,year,day,attendance,engagement_score,difficulty
2337,2024_803,2024,1,1,3.24,2
7036,2025_1100,2025,3,1,3.35,2
3668,2025_56,2025,2,1,4.08,4
4162,2025_213,2025,1,1,3.87,3
676,2024_269,2024,4,1,3.99,2


#### Generate final survey dataset

In [5]:
def generate_final_survey(daily):
    data = []
    
    grouped = daily.groupby("student_id")
    
    for student_id, group in grouped:
        attendance_rate = len(group) / 15
        
        completion = 1 if attendance_rate > 0.6 else 0
        
        satisfaction = np.clip(
            2 + attendance_rate * 3 + np.random.normal(0, 0.5), 1, 5
        )
        
        learning = np.clip(
            2 + attendance_rate * 3 + np.random.normal(0, 0.7), 1, 5
        )
        
        data.append({
            "student_id": student_id,
            "year": group["year"].iloc[0],
            "completion": completion,
            "satisfaction": round(satisfaction, 2),
            "self_reported_learning": round(learning, 2)
        })
    
    df = pd.DataFrame(data)

    #  a few missing rows (not everyone fills final survey)
    df = df.sample(frac=0.97, random_state=42)
    
    return df

final = generate_final_survey(daily)
final.to_csv("../data/raw/final_survey.csv", index=False)

final.head()

,student_id,year,completion,satisfaction,self_reported_learning
1079,2025_317,2025,0,2.13,2.05
405,2024_5,2024,0,2.73,3.39
1491,2025_872,2025,0,2.16,1.83
239,2024_27,2024,1,4.99,3.48
610,2024_773,2024,1,4.26,4.15


#### Sanity checks

In [6]:
print("Applications shape:", applications.shape)
print("Daily shape:", daily.shape)
print("Final shape:", final.shape)

print("\nSample countries:")
print(applications["country"].value_counts().head())

print("\nDisability distribution:")
print(applications["disability_type"].value_counts())

Applications shape: (2450, 8)
Daily shape: (7143, 6)
Final shape: (1540, 5)

Sample countries:
country
Peru       374
Kenya      351
India      346
Germany    337
UK         298
Name: count, dtype: int64

Disability distribution:
disability_type
none                 1833
cognitive             125
hearing               124
visual                122
motor                  95
multiple               83
prefer_not_to_say      68
Name: count, dtype: int64
